In [1]:
import pandas as pd
import numpy as np

from behavioral_ml.etl import load_data
from behavioral_ml.features import mouse, key
from behavioral_ml import train, evaluation

mouse_df, key_df = load_data("../data/raw_kmt_dataset")

In [2]:
basic_key_features = (
    key_df
    .pipe(key.add_key_dynamics)
    .pipe(key.aggregate_basic_key_dynamics)
)

COUNT_KEY_GROUPS = {
    "numpad": key.NUMPAD_KEYS,
    "tabs": key.TAB_KEYS,
    "correction": key.CORRECTION_KEYS,
    "enter": ["enter"]
}

count_key_features = (
    key_df
    .pipe(key.aggregate_selected_key_counts, key_groups=COUNT_KEY_GROUPS)
)

DYNAMICS_KEY_GROUPS = {
    "shifts": key.SHIFT_KEYS
}
FEATURES = ["Dwell Time", "Flight Time", "UU"]

selected_key_dynamics = (
    key_df
    .pipe(key.add_key_dynamics)
    .pipe(key.aggregate_selected_key_dynamics, key_groups=DYNAMICS_KEY_GROUPS, features=FEATURES)
)

key_features = (
    basic_key_features
    .join(count_key_features)
    .join(selected_key_dynamics)
)

key_features.head()

Dwell_median  Dwell_iqr  Flight_median  \
User ID Is Genuine Test ID                                           
1       False      1            0.075183   0.024578       0.152125   
                   2            0.086801   0.014753       0.379757   
                   3            0.073625   0.025564       0.127848   
                   4            0.121413   0.021379       0.085070   
                   5            0.170910   0.049758       0.060691   

                            Flight_iqr  DD_median    DD_iqr  UU_median  \
User ID Is Genuine Test ID                                               
1       False      1          0.231739   0.215051  0.294208   0.216582   
                   2          0.552763   0.480338  0.689027   0.468952   
                   3          0.405288   0.206717  0.433331   0.187322   
                   4          0.805698   0.220963  0.819443   0.206442   
                   5          0.299547   0.237161  0.484073   0.184446   

                              UU_iqr  numpad_count  tabs_count  \
User ID Is Genuine Test ID                                       
1       False      1        0.320033            46           0   
                   2        0.786050             0           0   
                   3        0.421998            50           2   
                   4        1.025717             0           0   
                   5        0.218457             0           5   

                            correction_count  enter_count  \
User ID Is Genuine Test ID                                  
1       False      1                       0            0   
                   2                       0            0   
                   3                       4            0   
                   4                       0            0   
                   5                       6            0   

                            shifts_Dwell Time_mean  \
User ID Is Genuine Test ID                           
1       False      1                      0.192708   
                   2                      0.675441   
                   3                           NaN   
                   4                      0.321364   
                   5                      0.379624   

                            shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>  \
User ID Is Genuine Test ID                                                                                                  
1       False      1                                                 0.025738                                               
                   2                                                 0.353099                                               
                   3                                                      NaN                                               
                   4                                                 0.013131                                               
                   5                                                 0.061078                                               

                            shifts_Flight Time_mean  \
User ID Is Genuine Test ID                            
1       False      1                      -0.087519   
                   2                      -0.095782   
                   3                            NaN   
                   4                      -0.105524   
                   5                      -0.171269   

                            shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>  \
User ID Is Genuine Test ID                                                                                                   
1       False      1                                                 0.025463                                                
                   2                                                 0.026924                               

In [3]:
kinematic_mouse_features = (
    mouse_df
    .pipe(mouse.add_mouse_kinematics)
    .pipe(mouse.aggregate_movement_features)
)

button_mouse_features = (
    mouse_df
    .pipe(mouse.add_button_features)
    .pipe(mouse.aggregate_button_features)
)

mouse_features = (
    kinematic_mouse_features
    .join(button_mouse_features, how="outer")
)

mouse_features.head()

Path_median    Path_iqr  Duration_median  \
User ID Is Genuine Test ID                                             
1       False      1         288.162445  440.753629         0.808292   
                   2         156.540442  126.153846         1.186481   
                   3         506.975291  390.374930         1.560235   
                   4         147.184662  132.460121         0.715755   
                   5         180.208823  170.316779         0.503478   

                            Duration_iqr  V_real_mean  V_real_std  V_q10_mean  \
User ID Is Genuine Test ID                                                      
1       False      1            1.320218   247.238431  183.385716   44.552617   
                   2            0.891805   180.819024   91.076198   54.389009   
                   3             1.10403   332.795623  111.181444   94.278052   
                   4            0.388159   270.479865  108.191208   73.045616   
                   5            0.907254    323.70584  264.607996   83.026475   

                            V_median_mean   V_q90_mean  Straightness_mean  \
User ID Is Genuine Test ID                                                  
1       False      1           219.189709   1005.58684           0.621676   
                   2            213.97934   614.472911           0.613355   
                   3           400.809091  1034.360892           0.483614   
                   4           272.321455   813.464199           0.747512   
                   5           315.672675   990.003793           0.721954   

                            ...  Straightness_median    A_min_mean  \
User ID Is Genuine Test ID  ...                                      
1       False      1        ...             0.518662 -35474.536212   
                   2        ...             0.680645 -21564.360267   
                   3        ...             0.444299  -38270.97847   
                   4        ...             0.901379 -24680.760742   
                   5        ...             0.821403 -34156.522987   

                             A_plus_mean   Jerk_min_mean  Jerk_plus_mean  \
User ID Is Genuine Test ID                                                 
1       False      1        29032.348312 -4830879.387616  4603405.381208   
                   2        23625.258934 -3493473.695546  2988733.176243   
                   3        37162.957878 -5594145.315441  5780988.700273   
                   4         22708.88576 -3026919.551201  3722555.070636   
                   5         104635.3775  -10917781.4193  65237131.70116   

                            Movement_count  Burst_count  \
User ID Is Genuine Test ID                                
1       False      1                     7            0   
                   2                     7            0   
                   3                     5            0   
                   4                     8            0   
                   5                     9            0   

                            Click_Dwell_Time_mean  Click_Dwell_Time_std  \
User ID Is Genuine Test ID                                                
1       False      1                     0.102698              0.021886   
                   2                     0.133980              0.010070   
                   3                     0.095714              0.019424   
                   4                     0.220024              0.010642   
                   5                     0.167220              0.046395   

                            Click_count  
User ID Is Genuine Test ID               
1       False      1                  6  
                   2                  6  
                   3                  5  
                   4                  6  
                   5                  3  

[5 rows x 21 columns]

In [4]:
X, y = train.split_for_binary(key_features)

In [5]:
from sklearn.metrics import roc_auc_score

metrics = {
    "roc_auc_score": roc_auc_score,
    "metrics": evaluation.get_metrics
}

In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(C=1),
)

y_true, y_score, results = train.train_binary_classifier(model, cv, X, y, metrics=metrics)

/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_UU_mean'
 'shifts_UU_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dyna

In [7]:
aucs = [results[user]["roc_auc_score"] for user in results]
np.mean(aucs)

np.float64(0.8912500000000001)

In [8]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics.pairwise import manhattan_distances

X_enrolls, X_evals, y_evals = train.split_for_novelty(key_features, enroll=5)

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

y_true = []
y_score = []
roc_aucs = []
for user_id in range(1, 89):
    X_enroll = pipeline.fit_transform(X_enrolls[user_id])
    X_eval = pipeline.transform(X_evals[user_id])

    template = np.median(X_enroll, axis=0, keepdims=True)

    score = -manhattan_distances(X_eval, template).ravel()
    
    y_true.extend(y_evals[user_id])
    y_score.extend(score)

    roc_aucs.append(roc_auc_score(y_evals[user_id], score))

np.mean(roc_aucs), np.median(roc_aucs), np.std(roc_aucs)

/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_UU_mean'
 'shifts_UU_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dyna

(np.float64(0.9393181818181818),
 np.float64(0.98),
 np.float64(0.09373312520304962))

In [9]:
from sklearn.base import clone

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

X_evals, y_evals = train.get_differential_dataset(key_features, by="user")

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(C=1),
)

_, _, metrics = train.train_binary_classifier(model, cv, X_evals, y_evals, metrics={"roc_auc": roc_auc_score})

pd.DataFrame.from_dict(metrics, orient="index").mean()

/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_UU_mean'
 'shifts_UU_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dyna

accuracy    0.857576
roc_auc     0.906818
dtype: float64

In [10]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC

X, y = train.get_differential_dataset(key_features)
groups = X.index.get_level_values("User ID")


cv = StratifiedGroupKFold(n_splits=5)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SVC(kernel="rbf", gamma="auto"),
)

y_true = []
y_score = []
results = {}
accuracy_scores = []
roc_aucs = []
    
for train_idx, test_idx in cv.split(X, y, groups):
    model_fold = clone(model)
    model_fold.fit(X.iloc[train_idx], y[train_idx])
    
    score = model_fold.decision_function(X.iloc[test_idx])
    true = y[test_idx]
    accuracy_scores.append(model_fold.score(X.iloc[test_idx], y[test_idx]))
    roc_aucs.append(roc_auc_score(true, score))

    y_score.append(score)
    y_true.append(true)

accuracy_scores, roc_aucs

([0.8888888888888888,
  0.8703703703703703,
  0.8705882352941177,
  0.8588235294117647,
  0.8703703703703703],
 [0.9261728395061728,
  0.9281481481481482,
  0.9258823529411765,
  0.9326643598615918,
  0.9199999999999999])

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

X, y = train.get_differential_dataset(key_features)
groups = X.index.get_level_values("User ID")


cv = StratifiedGroupKFold(n_splits=5)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    LogisticRegression(),
)

y_true = []
y_score = []
results = {}
accuracy_scores = []
roc_aucs = []
    
for train_idx, test_idx in cv.split(X, y, groups):
    model_fold = clone(model)
    model_fold.fit(X.iloc[train_idx], y[train_idx])
    
    score = model_fold.predict_proba(X.iloc[test_idx])[:, 1]
    true = y[test_idx]
    accuracy_scores.append(model_fold.score(X.iloc[test_idx], y[test_idx]))
    roc_aucs.append(roc_auc_score(true, score))

    y_score.append(score)
    y_true.append(true)

accuracy_scores, roc_aucs

([0.5814814814814815,
  0.5703703703703704,
  0.6235294117647059,
  0.6509803921568628,
  0.6222222222222222],
 [0.5409876543209876,
  0.5820987654320988,
  0.6683044982698962,
  0.6155709342560554,
  0.6251234567901235])

In [12]:
from sklearn.svm import SVC

X, y = train.get_differential_dataset(key_features)

from sklearn.model_selection import StratifiedGroupKFold

groups = X.index.get_level_values("User ID")


cv = StratifiedGroupKFold(n_splits=5)

model = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    SVC(),
)

y_true = []
y_score = []
results = {}
accuracy_scores = []
roc_aucs = []
    
for train_idx, test_idx in cv.split(X, y, groups):
    model_fold = clone(model)
    model_fold.fit(X.iloc[train_idx], y[train_idx])
    
    score = model_fold.decision_function(X.iloc[test_idx])
    true = y[test_idx]
    accuracy_scores.append(model_fold.score(X.iloc[test_idx], y[test_idx]))
    roc_aucs.append(roc_auc_score(true, score))

    y_score.append(score)
    y_true.append(true)

accuracy_scores, roc_aucs

([0.8888888888888888,
  0.8703703703703703,
  0.8705882352941177,
  0.8588235294117647,
  0.8703703703703703],
 [0.9261728395061728,
  0.9281481481481482,
  0.9258823529411765,
  0.9326643598615918,
  0.9199999999999999])

In [17]:
X_enrolls, X_evals, y_evals = train.split_for_novelty(key_features)

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

y_true, y_score = train.train_anomaly_detector(pipeline, manhattan_distances, X_enrolls, X_evals, y_evals)

roc_auc_score(y_true, y_score)

/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_UU_mean'
 'shifts_UU_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dyna

0.9105862603305787

In [37]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

X_enrolls, X_evals, y_evals = train.split_for_novelty(key_features.join(mouse_features), enroll=5)

metrics = {
    "roc_auc_score": roc_auc_score,
    "eer": evaluation.get_eer_from_scores
}

preprocessing_pipeline = make_pipeline(
    SimpleImputer(strategy="median")
)

clf = IsolationForest(n_estimators=200)

y_true = []
y_score = []
results = []
for user_id in range(1, 89):
    
    X_enroll = preprocessing_pipeline.fit_transform(X_enrolls[user_id])
    X_eval = preprocessing_pipeline.transform(X_evals[user_id])

    clf.fit(X_enroll)
    
    score = clf.decision_function(X_eval)
       
    y_true.append(y_evals[user_id])
    y_score.append(score)


    if metrics != None:
        user_results = {}
        for m in metrics:
            user_results[m] = metrics[m](y_evals[user_id], score)
            results.append(user_results)

pd.DataFrame(results).agg(["mean", "median", "std"])

/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_UU_mean'
 'shifts_UU_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dyna

,roc_auc_score,eer
mean,0.821023,0.225568
median,0.840000,0.200000
std,0.154328,0.170209


In [35]:
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM

X_enrolls, X_evals, y_evals = train.split_for_novelty(key_features, enroll=5)

metrics = {
    "roc_auc_score": roc_auc_score,
    "eer": evaluation.get_eer_from_scores
}

preprocessing_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

clf = OneClassSVM(kernel="rbf", gamma="scale")

y_true = []
y_score = []
results = []
for user_id in range(1, 89):
    
    X_enroll = preprocessing_pipeline.fit_transform(X_enrolls[user_id])
    X_eval = preprocessing_pipeline.transform(X_evals[user_id])

    clf.fit(X_enroll)
    
    score = clf.decision_function(X_eval)
       
    y_true.append(y_evals[user_id])
    y_score.append(score)


    if metrics != None:
        user_results = {}
        for m in metrics:
            user_results[m] = metrics[m](y_evals[user_id], score)
            results.append(user_results)



pd.DataFrame(results).agg(["mean", "median", "std"])

/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_UU_mean'
 'shifts_UU_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(
/Volumes/Workspace/01_Projects/behavioral-ml/.venv/lib/python3.13/site-packages/sklearn/impute/_base.py:647: UserWarning: Skipping features without any observed values: ['shifts_Dwell Time_mean'
 'shifts_Dwell Time_<function aggregate_selected_key_dynamics.<locals>.<lambda> at 0x11dfee0c0>'
 'shifts_Flight Time_mean'
 'shifts_Flight Time_<function aggregate_selected_key_dyna

,roc_auc_score,eer
mean,0.942727,0.110227
median,0.980000,0.075000
std,0.074584,0.122161
